# 🚀 Routing101 (Profile 1152d) - Multi-Signal Video Retrieval on Kaggle

Notebook này được tối ưu hoá để chạy **Web App Tìm Kiếm Video Đa Tín Hiệu (FastAPI + Static UI)** với mô hình **SigLIP2 SO400M (1152-dim)** mới nhất (`google/siglip2-so400m-patch14-384`):
- **Embedding Profile**: `1152` (Dimension 1152, Chunked Summaries, ASR Segment Timestamp resolution)
- **Repo Branch**: Nhánh `main` từ `https://github.com/hoangducbao/UnoptimalLonx.git` (Được đồng bộ đầy đủ từ `Routing101` kèm các bản vá tối ưu hóa môi trường Kaggle).
- **Dataset hỗ trợ**:
  - `https://www.kaggle.com/datasets/nguyenthanghuu/aic2026-dataset` (`Keyframes`, `Videos`, `map-keyframes`)
  - Dataset chứa embedding 1152-dim (`1152keyframe`, `1152caption`, `1152transcript`, `1152summary`) hoặc bundle trích xuất tương ứng.

### ⚙️ Cấu hình bắt buộc trên thanh công cụ Kaggle:
1. **Accelerator**: Chọn **`GPU T4 x2`** hoặc GPU P100 / CPU.
2. **Internet**: Chọn `ON` (để tải model SigLIP2 Giant ~1.7 GB và mở Cloudflare Tunnel).
3. **Data > Input**: Đã thêm dataset AIC / Embeddings tương ứng.


## 📦 BƯỚC 1: Tải Mã Nguồn Repo (Nhánh 1532) & Cài đặt Dependencies

In [ ]:
import os
import sys
import shutil
from pathlib import Path

# 0. Luôn đưa cwd về /kaggle/working trước để tránh lỗi 'deleted working directory'
try:
    os.chdir("/kaggle/working")
except Exception:
    pass
%cd /kaggle/working

WORKSPACE_DIR = Path("/kaggle/working/Routing101")
REPO_URL = "https://github.com/hoangducbao/UnoptimalLonx.git"
BRANCH = "main"

# 1. Clone mã nguồn Routing101 về Kaggle theo nhánh main
if not (WORKSPACE_DIR / "backend").exists():
    print(f"📥 Đang tải mã nguồn từ {REPO_URL} (nhánh {BRANCH})...")
    if WORKSPACE_DIR.exists():
        shutil.rmtree(WORKSPACE_DIR, ignore_errors=True)
    !git clone -b {BRANCH} {REPO_URL} /kaggle/working/Routing101
else:
    print(f"🔄 Đang đồng bộ và cập nhật mã nguồn mới nhất từ nhánh {BRANCH}...")
    !cd /kaggle/working/Routing101 && git remote set-url origin {REPO_URL} && git fetch origin {BRANCH} && git checkout {BRANCH} && git reset --hard origin/{BRANCH}

# 2. Chuyển vào thư mục repository và thiết lập sys.path
%cd /kaggle/working/Routing101
try:
    os.chdir("/kaggle/working/Routing101")
except Exception:
    pass

if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))

# 3. Cài đặt các thư viện cần thiết
!pip install -q --upgrade pip
!pip install -q fastapi uvicorn python-multipart cachetools "elasticsearch>=8.11,<8.13" faiss-cpu
!pip install -q transformers pillow tqdm pandas numpy
!pip install -q pycloudflared python-dotenv requests ultralytics

print("✅ Bước 1: Mã nguồn nhánh main đã sẵn sàng & Dependencies đã được cài đặt!")

## 🐘 BƯỚC 2: Cài đặt và khởi chạy Elasticsearch nền (Không cần Docker)

In [ ]:
import os
import time
import requests
import subprocess
from pathlib import Path

print("=" * 60)
print("🐘 CÀI ĐẶT VÀ KHỞI ĐỘNG ELASTICSEARCH NỀN")
print("=" * 60)

# 1. Dọn dẹp sạch tiến trình cũ & file lock
!pkill -9 -f elasticsearch 2>/dev/null || true
!rm -f /opt/elasticsearch-8.11.0/data/node.lock 2>/dev/null || true

# 2. Tải và giải nén Elasticsearch 8.11.0 (nếu chưa tải)
if not Path("/opt/elasticsearch-8.11.0").exists():
    print("📥 Đang tải Elasticsearch 8.11.0 (khoảng 30s)...")
    !wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-8.11.0-linux-x86_64.tar.gz
    print("📦 Đang giải nén vào /opt/...")
    !tar -xzf elasticsearch-8.11.0-linux-x86_64.tar.gz -C /opt/
    !rm -f elasticsearch-8.11.0-linux-x86_64.tar.gz

# 3. Cấu hình Development Mode an toàn (IPv4 127.0.0.1 để bypass bootstrap check)
es_yml_path = Path("/opt/elasticsearch-8.11.0/config/elasticsearch.yml")
es_config = '''cluster.name: routing101-cluster
node.name: node-1
network.host: 127.0.0.1
http.port: 9200
discovery.type: single-node
xpack.security.enabled: false
xpack.security.enrollment.enabled: false
'''
es_yml_path.write_text(es_config, encoding="utf-8")

# 4. Phân quyền cho user esuser (Elasticsearch không cho phép chạy dưới quyền root)
!useradd -m -s /bin/bash esuser 2>/dev/null || true
!chown -R esuser:esuser /opt/elasticsearch-8.11.0
!touch /kaggle/working/elasticsearch.log && chmod 666 /kaggle/working/elasticsearch.log

# 5. Khởi động Elasticsearch nền và lưu log trực tiếp vào /kaggle/working/elasticsearch.log
print("🚀 Đang khởi động Elasticsearch service...")
es_log_file = open("/kaggle/working/elasticsearch.log", "a")
es_proc = subprocess.Popen(
    ["su", "esuser", "-c", "/opt/elasticsearch-8.11.0/bin/elasticsearch"],
    stdout=es_log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# 6. Đợi đến khi Elasticsearch sẵn sàng nhận request qua cổng 9200
es_ready = False
for i in range(45):
    try:
        r = requests.get("http://127.0.0.1:9200", timeout=1)
        if r.status_code == 200:
            es_ready = True
            print(f"\n✅ Elasticsearch đã khởi động thành công và sẵn sàng tại http://127.0.0.1:9200! (sau {i+1}s)")
            print(f"📄 File log Elasticsearch: /kaggle/working/elasticsearch.log")
            break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(1)

if not es_ready:
    print("\n\n❌ Elasticsearch chưa sẵn sàng. Log chi tiết từ /kaggle/working/elasticsearch.log:")
    print("-" * 60)
    !cat /kaggle/working/elasticsearch.log | tail -n 35
    print("-" * 60)

## ⚙️ BƯỚC 3: Cấu hình Profile 1152 (SO400M) & Tự Động Nhận Diện Dataset

In [ ]:
import os
import csv
import numpy as np
from pathlib import Path

# Thiết lập biến môi trường chỉ định profile 1152
os.environ["R101_EMBED"] = "1152"

KAGGLE_INPUT = Path("/kaggle/input")
all_found_dirs = {}
found_1152_dirs = {}

print("=" * 70)
print("🔍 QUÉT VÀ NHẬN DIỆN DỮ LIỆU PROFILE 1152 TRONG /kaggle/input/...")
print("=" * 70)

LEAF_NAMES = {
    "keyframes", "videos", "video", "degarr", "map-keyframes", "map_keyframes",
    "ocr", "captions", "caption", "transcripts", "transcript", "summaries", "summary",
    "filtered_object", "filtered_objects", "objects", "object_detection"
}

if KAGGLE_INPUT.exists():
    for root, dirs, _ in os.walk(KAGGLE_INPUT):
        root_path = Path(root)
        name_lower = root_path.name.lower()
        all_found_dirs[name_lower] = root_path
        rel = root_path.relative_to(KAGGLE_INPUT)
        depth = len(rel.parts)
        
        # Kiểm tra file .npy trong thư mục để tự động nhận diện vector dimension
        npy_files = list(root_path.glob("*.npy"))
        is_1152 = False
        if npy_files:
            try:
                sample_vec = np.load(npy_files[0])
                if sample_vec.shape[-1] == 1152:
                    found_1152_dirs[name_lower] = root_path
                    is_1152 = True
            except Exception:
                pass

        if 0 < depth <= 5:
            indent = "   " * (depth - 1)
            tag = f" 🎯 [{len(npy_files)} files 1152-dim]" if is_1152 else ""
            print(f"{indent}└── 📁 {root_path.name}{tag}")

        # Nếu là thư mục leaf hoặc đã tìm thấy 1152 vectors thì không duyệt sâu thêm
        if is_1152 or name_lower in LEAF_NAMES or depth >= 6:
            dirs.clear()
        else:
            dirs[:] = [d for d in dirs if not d.startswith(("L0", "L1", "L2", "v_", "V_"))]
else:
    print("⚠️ Không tìm thấy thư mục /kaggle/input/")

print("=" * 70)

def get_target(keys: list, fallback: Path) -> Path:
    for k in keys:
        if k.lower() in all_found_dirs:
            return all_found_dirs[k.lower()]
    return fallback

# 1. Dữ liệu thô (Keyframes, Videos, map-keyframes)
KEYFRAMES_DIR = get_target(["keyframes"], KAGGLE_INPUT / "datasets/nguyenthanghuu/aic2026-dataset/Keyframes")
VIDEOS_DIR = get_target(["videos", "video", "degarr"], KAGGLE_INPUT / "datasets/nguyenthanghuu/aic2026-dataset/Videos")
MAP_KEYFRAMES_DIR = get_target(["map-keyframes", "map_keyframes"], KAGGLE_INPUT / "datasets/nguyenthanghuu/aic2026-dataset/map-keyframes")

if not KEYFRAMES_DIR.exists():
    KEYFRAMES_DIR = Path("/kaggle/working/keyframes_dummy")
    KEYFRAMES_DIR.mkdir(parents=True, exist_ok=True)
if not VIDEOS_DIR.exists():
    VIDEOS_DIR = Path("/kaggle/working/videos_dummy")
    VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Nhận diện SigLIP2 1152 Frame Embeddings
SIGLIP_DIR = None
for k in ["1152keyframe", "keyframe_1152", "1152embed", "1152", "keyframe"]:
    if k in found_1152_dirs:
        SIGLIP_DIR = found_1152_dirs[k]
        break
if not SIGLIP_DIR:
    for k, p in found_1152_dirs.items():
        if "transcript" not in k and "caption" not in k and "summary" not in k:
            SIGLIP_DIR = p
            break
if not SIGLIP_DIR:
    SIGLIP_DIR = get_target(["1152keyframe", "1152embed"], KAGGLE_INPUT / "1152embed/1152keyframe")

# Nếu thư mục được chọn có thư mục con 1152keyframe, trỏ vào sâu hơn
if not list(SIGLIP_DIR.glob("*.npy")):
    if (SIGLIP_DIR / "1152keyframe").exists():
        SIGLIP_DIR = SIGLIP_DIR / "1152keyframe"

# 3. Các embedding phụ trợ (Transcript, Caption, Summary)
# LƯU Ý: Tuyệt đối KHÔNG lấy nhầm embeddings 768/1152 từ rrqbundle!
def resolve_1152_aux(tag: str) -> Path:
    for k in [f"1152{tag}", f"{tag}_1152", tag]:
        if k in found_1152_dirs:
            return found_1152_dirs[k]
    # Kiểm tra bên trong 1152embed hoặc thư mục cha của SIGLIP_DIR
    parent_candidates = [
        all_found_dirs.get("1152embed"),
        SIGLIP_DIR.parent if SIGLIP_DIR else None,
        all_found_dirs.get("thangcholenguyn")
    ]
    for parent in parent_candidates:
        if parent:
            for sub in [f"1152{tag}", f"{tag}_1152", tag]:
                p = parent / sub
                if p.exists() and list(p.glob("*.npy")):
                    try:
                        if np.load(list(p.glob("*.npy"))[0]).shape[-1] == 1152:
                            return p
                    except Exception:
                        pass
    empty_dir = Path(f"/kaggle/working/empty_1152/{tag}")
    empty_dir.mkdir(parents=True, exist_ok=True)
    return empty_dir

TRANSCRIPT_EMBED_DIR = resolve_1152_aux("transcript")
CAPTION_EMBED_DIR = resolve_1152_aux("caption")
SUMMARY_EMBED_DIR = resolve_1152_aux("summary")

# Dữ liệu văn bản (Captions, Transcripts, Summaries, OCR)
CAPTIONS_DIR = get_target(["captions", "caption"], KAGGLE_INPUT / "captions")
OCR_DIR = get_target(["ocr"], KAGGLE_INPUT / "ocr")
SUMMARIES_DIR = get_target(["summaries", "summary"], KAGGLE_INPUT / "summaries")
TRANSCRIPTS_DIR = get_target(["transcripts", "transcript"], KAGGLE_INPUT / "transcripts")

# 4. Object Detection (OD) Filter
FILTERED_OBJECT_DIR = get_target(["filtered_object", "filtered_objects", "objects", "object_detection"], KAGGLE_INPUT / "filtered_object")
CLASS_VOCAB_CSV = FILTERED_OBJECT_DIR / "class_vocab.csv" if FILTERED_OBJECT_DIR.exists() else Path("/kaggle/working/class_vocab.csv")

if FILTERED_OBJECT_DIR.exists() and not CLASS_VOCAB_CSV.exists():
    print("🔨 Đang tự động tạo class_vocab.csv từ filtered_object/*.csv...")
    names = set()
    for p in FILTERED_OBJECT_DIR.glob("*.csv"):
        if p.name == "class_vocab.csv": continue
        try:
            with open(p, "r", encoding="utf-8", newline="") as f:
                reader = csv.DictReader(f)
                if reader.fieldnames and "class_name" in reader.fieldnames:
                    for row in reader:
                        raw = row.get("class_name")
                        if raw:
                            names.add(" ".join(str(raw).strip().lower().split()))
        except Exception:
            continue
    CLASS_VOCAB_CSV = Path("/kaggle/working/class_vocab.csv")
    with open(CLASS_VOCAB_CSV, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["class_name"])
        for name in sorted(names):
            writer.writerow([name])
    print(f"✅ Đã tạo {len(names)} unique OD classes tại {CLASS_VOCAB_CSV}")

# Đảm bảo thư mục lưu index 1152 và summary embed trên ổ ghi được
Path("/kaggle/working/index/1152").mkdir(parents=True, exist_ok=True)
if not SUMMARY_EMBED_DIR.exists():
    SUMMARY_EMBED_DIR = Path("/kaggle/working/summary_embed")
    SUMMARY_EMBED_DIR.mkdir(parents=True, exist_ok=True)

# 4. Ghi đè file backend/config.py với Profile 1152 (SigLIP2 Giant)
config_py_content = f'''import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

IS_KAGGLE = True
FETCH_K = 100
DISPLAY_N = 200
RRF_K = 60
NEIGHBOR_WINDOW = 7
TOP_G_DEFAULT = 10

DATASET_MODE = "AIC"
EMBED_PROFILE = "1152"
EMBED_DIM = 1152
SUMMARY_CHUNKED = True
SIGLIP2_MODEL_ID = "google/siglip2-so400m-patch14-384"

FRAME_SIGLIP2_GLOB = "{SIGLIP_DIR}/*.npy"

ASR_EMBED_DIR = Path("{TRANSCRIPT_EMBED_DIR}")
TRANSCRIPTS_DIR = Path("{TRANSCRIPTS_DIR}")

CAPTIONING_DIR = Path("{CAPTIONS_DIR}")
SIGLIP_CAPTION_DIR = Path("{CAPTION_EMBED_DIR}")

OCR_DIR = Path("{OCR_DIR}")

FILTERED_OBJECT_DIR = Path("{FILTERED_OBJECT_DIR}")
CLASS_VOCAB_CSV = Path("{CLASS_VOCAB_CSV}")

SUMMARY_DIR = Path("{SUMMARIES_DIR}")
SUMMARY_EMBED_DIR = Path("{SUMMARY_EMBED_DIR}")

MAP_KEYFRAMES_DIR = Path("{MAP_KEYFRAMES_DIR}")
THUMBNAIL_ROOT = Path("{KEYFRAMES_DIR}")
VIDEO_DIR = Path("{VIDEOS_DIR}")

INDEX_PREFIX = "routing101"

try:
    SUMMARY_EMBED_DIR.mkdir(parents=True, exist_ok=True)
except Exception:
    pass

REPO_ROOT = Path(__file__).resolve().parent.parent
INDEX_DIR = Path("/kaggle/working/index/1152")
INDEX_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_DIR = REPO_ROOT / "pipeline"
ASR_INDEX_DIR = INDEX_DIR / f"{{INDEX_PREFIX}}_asr"
CAPTION_INDEX_DIR = INDEX_DIR / f"{{INDEX_PREFIX}}_caption"
SUMMARY_INDEX_DIR = INDEX_DIR / f"{{INDEX_PREFIX}}_summary"
ASR_INDEX_DIR.mkdir(parents=True, exist_ok=True)
CAPTION_INDEX_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_INDEX_DIR.mkdir(parents=True, exist_ok=True)
SIGLIP_ASR_FAISS = ASR_INDEX_DIR / "siglip_asr_flat_ip.index"
SIGLIP_ASR_META = ASR_INDEX_DIR / "meta_siglip_asr.csv"
SIGLIP_CAPTION_FAISS = CAPTION_INDEX_DIR / "siglip_caption_flat_ip.index"
SIGLIP_CAPTION_META = CAPTION_INDEX_DIR / "meta_siglip_caption.csv"
SIGLIP_SUMMARY_FAISS = SUMMARY_INDEX_DIR / "siglip_summary_flat_ip.index"
SIGLIP_SUMMARY_META = SUMMARY_INDEX_DIR / "meta_siglip_summary.csv"

ES_HOST = "http://127.0.0.1:9200"
ES_INDEX_ASR = "asr_segments"
ES_INDEX_CAPTION = "caption_frames"
ES_INDEX_OCR = "ocr_frames"
ES_INDEX_SUMMARY = "summary_videos"

CPU_BUDGET = max(1, (os.cpu_count() or 4) - 2)

def tune_thread_pools(device: str) -> None:
    import faiss
    import torch
    if device == "cpu":
        torch.set_num_threads(CPU_BUDGET)
    torch.set_num_interop_threads(1)
    faiss.omp_set_num_threads(CPU_BUDGET)
'''

Path("/kaggle/working/Routing101/backend/config.py").write_text(config_py_content, encoding="utf-8")
print("✅ Đã thiết lập cấu hình Profile 1152 vào /kaggle/working/Routing101/backend/config.py!")

# 5. Hiển thị trạng thái
status = lambda p: "✅ SẴN SÀNG" if p.exists() else "⚠️ KHÔNG TÌM THẤY"
print("=" * 70)
print(f"• Profile                : 1152 (google/siglip2-so400m-patch14-384)")
print(f"• Keyframes Root         : {status(KEYFRAMES_DIR)} -> {KEYFRAMES_DIR}")
print(f"• Videos Root            : {status(VIDEOS_DIR)} -> {VIDEOS_DIR}")
print(f"• Map-Keyframes CSV      : {status(MAP_KEYFRAMES_DIR)} -> {MAP_KEYFRAMES_DIR}")
print(f"• 1152 Embeddings (.npy) : {status(SIGLIP_DIR)} -> {SIGLIP_DIR}")
print(f"• Captions CSV           : {status(CAPTIONS_DIR)} -> {CAPTIONS_DIR}")
print(f"• Caption Embeddings     : {status(CAPTION_EMBED_DIR)} -> {CAPTION_EMBED_DIR}")
print(f"• OCR CSV                : {status(OCR_DIR)} -> {OCR_DIR}")
print(f"• Object Detection (OD)  : {status(FILTERED_OBJECT_DIR)} -> {FILTERED_OBJECT_DIR}")
print(f"• OD Class Vocabulary    : {status(CLASS_VOCAB_CSV)} -> {CLASS_VOCAB_CSV}")
print(f"• Transcripts CSV        : {status(TRANSCRIPTS_DIR)} -> {TRANSCRIPTS_DIR}")
print(f"• Transcript Embeddings  : {status(TRANSCRIPT_EMBED_DIR)} -> {TRANSCRIPT_EMBED_DIR}")
print(f"• Summaries TXT          : {status(SUMMARIES_DIR)} -> {SUMMARIES_DIR}")
print(f"• Summary Embeddings     : {status(SUMMARY_EMBED_DIR)} -> {SUMMARY_EMBED_DIR}")
print("=" * 70)

## 🧪 BƯỚC 4: Kiểm tra Kết Nối Elasticsearch & SigLIP2 Search

In [ ]:
import requests
import time
import importlib

print("=" * 60)
print("🔍 KIỂM TRA HỆ THỐNG TRƯỚC KHI CHẠY BACKEND")
print("=" * 60)

# Reload config và es_client
import backend.config
importlib.reload(backend.config)
import backend.es_client
importlib.reload(backend.es_client)

# 1. Kiểm tra HTTP trực tiếp tới Elasticsearch
try:
    r = requests.get("http://127.0.0.1:9200", timeout=2)
    if r.status_code == 200:
        print("✅ [1/3] Elasticsearch (HTTP 127.0.0.1:9200): OK")
    else:
        print(f"⚠️ [1/3] Elasticsearch HTTP status: {r.status_code}")
except Exception as e:
    print(f"❌ [1/3] Không thể kết nối tới Elasticsearch: {e}")

# Kiểm tra qua Python client
try:
    es = backend.es_client.get_es_client(force_new=True)
    es_info = es.info()
    cluster = es_info.get('cluster_name', 'unknown')
    version = es_info.get('version', {}).get('number', 'unknown')
    print(f"✅ [2/3] Elasticsearch (Python Client): OK (Cluster: {cluster}, v{version})")
except Exception as e:
    print(f"⚠️ [2/3] Elasticsearch (Python Client): LỖI - {type(e).__name__}: {e}")
    print(f"   → Kiểm tra ES_HOST trong backend/config.py và đảm bảo dùng http://127.0.0.1:9200")

# 2. Kiểm tra tìm kiếm mẫu qua SigLIP2
try:
    from backend.search import keyframe as kf_mod
    t0 = time.time()
    sample_res = kf_mod.search_siglip2_frame("person riding a bicycle", k=5)
    dt = (time.time() - t0) * 1000
    print(f"✅ [3/3] SigLIP2 Vector Search: Thành công ({len(sample_res)} kết quả trong {dt:.1f}ms)")
except Exception as e:
    print(f"⚠️ [3/3] Search test warning: {e}")

print("=" * 60)
print("🎉 Kiểm tra hoàn tất, chuyển sang Bước 5 để mở Web App!")

## 🌐 BƯỚC 5: Khởi chạy Web App & Mở Public URL qua Cloudflare Tunnel

In [ ]:
import subprocess
import time
import re
import os
import requests
from pathlib import Path

%cd /kaggle/working/Routing101

# 1. Tải và cài đặt cloudflared tunnel client (nếu chưa có)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1 || true

# 2. Khởi động FastAPI Backend với Uvicorn (Profile 1152)
print("🚀 Đang khởi động FastAPI Backend (Routing101 - Profile 1152d)...")
print("   ⏳ Lần đầu chạy trên Kaggle, quá trình tải model weights ~1.7 GB và build FAISS indices")
print("      có thể mất từ 1-3 phút. Tiến trình [startup] sẽ được hiển thị trực tiếp bên dưới:")
print("   ☕ Hãy pha một tách cà phê và kiên nhẫn đợi...")

backend_log_path = "/kaggle/working/backend.log"
backend_log = open(backend_log_path, "w")

backend_env = os.environ.copy()
backend_env["R101_EMBED"] = "1152"

backend_proc = subprocess.Popen(
    ["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=backend_log,
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/Routing101",
    env=backend_env
)

# 3. Khởi chạy Cloudflare Tunnel để expose port 8000 ra Internet
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
start_t = time.time()
while time.time() - start_t < 30:
    line = tunnel_proc.stdout.readline()
    if not line:
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

# 4. HEALTH CHECK: Đợi đến khi Uvicorn hoàn tất nạp model (timeout 720s = 12 phút)
print("⏳ Đang đợi Backend tải xong model 1152-dim và mở cổng 8000...")
backend_ready = False
last_log_pos = 0
for attempt in range(720):
    if backend_proc.poll() is not None:
        print("\n❌ Backend đã dừng đột ngột! Xem log bên dưới:")
        print("-" * 60)
        print(open(backend_log_path).read()[-3000:])
        print("-" * 60)
        break
    try:
        resp = requests.get("http://127.0.0.1:8000/docs", timeout=1)
        if resp.status_code == 200:
            backend_ready = True
            break
    except Exception:
        pass
    # Hiển thị tiến trình từ backend.log
    if attempt > 0 and attempt % 5 == 0:
        try:
            with open(backend_log_path, "r") as lf:
                lf.seek(last_log_pos)
                new_lines = lf.read()
                last_log_pos = lf.tell()
                for l in new_lines.strip().split("\n"):
                    if "[startup]" in l:
                        print(f"   📌 {l.strip()}")
        except Exception:
            pass
    if attempt > 0 and attempt % 30 == 0:
        elapsed = int(time.time() - start_t)
        print(f"   ... đã đợi {elapsed}s, backend vẫn đang nạp weights/indexes...")
    time.sleep(1)

# 5. Hiển thị link truy cập
elapsed_total = int(time.time() - start_t)
if backend_ready and public_url:
    print("\n" + "=" * 70)
    print(f"🎉 WEB APP ROUTING101 (1152-DIM SO400M) ĐÃ SẴN SÀNG SAU {elapsed_total}s!")
    print("=" * 70)
    print(f"🔗 TRUY CẬP GIAO DIỆN TẠI:  {public_url}/app/")
    print(f"📑 API SWAGGER DOCS TẠI  :  {public_url}/docs")
    print("=" * 70 + "\n")
elif not public_url:
    print("❌ Không tạo được Cloudflare Tunnel. Hãy kiểm tra kết nối Internet trong cài đặt Notebook!")
else:
    print(f"⚠️ Backend chưa sẵn sàng sau {elapsed_total}s. Xem log cuối cùng:")
    print("-" * 60)
    print(open(backend_log_path).read()[-2000:])
    print("-" * 60)
    print("💡 Nếu log cho thấy vẫn đang load, hãy chạy Bước 6 để theo dõi tiếp!")

## 📜 BƯỚC 6: Xem Logs Trực Tiếp (Real-time Log Viewer)

Chạy cell dưới đây để theo dõi các truy vấn tìm kiếm, RRF fusion, và thời gian thực thi của backend theo thời gian thực:

In [ ]:
# Xem 50 dòng log gần nhất của backend
!tail -n 50 /kaggle/working/backend.log

# Vòng lặp stream log trực tiếp (Nhấn nút Stop/Interrupt trên thanh công cụ để dừng xem log)
try:
    with open("/kaggle/working/backend.log", "r") as f:
        f.seek(0, 2)  # Seek to end
        print("--- BẮT ĐẦU THEO DÕI LOGS (Nhấn Interrupt để thoát) ---")
        while True:
            line = f.readline()
            if line:
                print(line, end="")
            else:
                time.sleep(0.5)
except KeyboardInterrupt:
    print("\nĐã dừng theo dõi log. Backend vẫn tiếp tục chạy ngầm.")